# 09: GeoPandas - DataFrames with geometry for GIS applications


In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gp
from pathlib import Path
from shapely.geometry import Point, Polygon

In [ ]:
datapath = Path('data/geopandas/')


**[Nice overview tutorial](https://geopandas.org/en/stable/getting_started/introduction.html)**
**[Examples Gallery](https://geopandas.org/en/stable/gallery/index.html)**

### OVERVIEW
***
In this notebook we will explore how `geopandas` works with GIS data (e.g. shapefiles and the objects within them) using geometrically-aware DataFrames. We will also explore how common GIS-related operations are available in `geopandas`. This notebook relies on publicly available GIS datasets for the city of Madison, Wisconsin.

#### get some data - `read_file` is the ticket for GeoJSON, shapefiles, GDB, etc.

In [ ]:
parks = gp.read_file(datapath / 'Madison_Parks.geojson')

### writing back out is veeeery similar with `to_file` but give a few options for formats

In [ ]:
parks.to_file(datapath / "parks.shp")

In [ ]:
# or with a different driver
parks.to_file(datapath / "parks.json", driver='GeoJSON')

### this now looks like a Pandas DataFrame but there's a special column `geometry`

In [ ]:
parks.head()

### also some important metadata particularly the [CRS](https://en.wikipedia.org/wiki/Spatial_reference_system)

In [ ]:
parks.crs

> ### pro tip: You can have multiple geometry columns but only one is _active_ -- this is important later as we do operations on GeoDataFrames. The column labeled `geometry` is typically the active one but you [you can change it](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html).

### So what's up with these geometries? They are represented as [`shapely`](https://shapely.readthedocs.io/en/stable/manual.html) objects so can be:
- #### polygon / multi-polygon
- #### point / multi-point
- #### line / multi-line

### we can access with pandas `loc` and `iloc` references

In [ ]:
parks.iloc[1].geometry.geom_type

In [ ]:
parks.loc[parks.ShortName=='Olin'].geometry.geom_type

### There are other cool shapely properties like `area`

In [ ]:
parks.loc[parks.ShortName=='Olin'].geometry.area

### ruh-roh - what's up with this CRS and tiny area number?

In [ ]:
parks.crs

### area units in lat/long don't make sense. Let's project to something in meters (but how?) `to_crs` will do it, but importantly, either reassign or set `inplace=True`

In [ ]:
parks.to_crs(3071, inplace=True)
parks.crs

In [ ]:
parks.loc[parks.ShortName=='Olin'].geometry.area

### there are loads of useful methods for `shapely` objects for relationships between geometries (intersection, distance, etc.) but we will skip these for now because GeoPandas facilitates these things for entire geodataframes! #sick

> ### Pro Tip: There are occasions when, while connected to VPN, the first geometry in a GeoGataDrame has its CRS set to infinity rather than a target CRS. There are a couple potential fixes for that [here](https://gist.github.com/kallejahn/186496059f0d4bb0e9d6e6460c32055e)

## VISUALIZATION
***

### So back to GeoDataFrames.....we can look at them spatially as well with `plot()`

In [ ]:
parks.plot()

### easily make a chloropleth map using a selected column as the color (and add a legend) using `plot()`

In [ ]:
parks.columns

In [ ]:
parks.sample(4)

In [ ]:
parks.plot(column="Acreage", cmap = 'magma', k=5, legend=True, legend_kwds={'shrink': 0.6})

### also a very cool interactive plot options with a basemap using `explore()`

In [ ]:
parks.explore(column='Acreage', cmap='magma')

### we can read in another shapefile

In [ ]:
hoods = gp.read_file(datapath / 'Neighborhood_Associations.geojson')

In [ ]:
hoods.sample(5)

### and we can plot these on top of each other

In [ ]:
ax_hood = hoods.plot()
# now plot the other one but specify which axis to plot on (ax=ax_hood)
parks.plot(column="Acreage", cmap = 'magma', k=5, legend=True, legend_kwds={'shrink': 0.6}, ax=ax_hood)

### WAT! Why so far apart?

In [ ]:
hoods.crs

### we need to reproject. Geopandas uses `to_crs()` for this purpose

In [ ]:
# we can reproject, and set hoods to park crs 
hoods.to_crs(parks.crs, inplace=True)

In [ ]:
ax_hood = hoods.plot()
# now plot the other one but specify which axis to plot on (ax=ax_hood)
parks.plot(column="Acreage", cmap = 'magma', k=5, legend=True, legend_kwds={'shrink': 0.6}, ax=ax_hood)

### or similarly with the interactive maps

In [ ]:
m_hood = hoods.explore()
parks.explore(column="Acreage", cmap = 'magma', k=5, legend=True, legend_kwds={'shrink': 0.6}, m=m_hood)

### we can make a new geodataframe using shapely properties of the geometry - how about centroids?

## TEST YOUR SKILLS #0
- make a new geodataframe of the parks
- add a columns with centroids for each park
- plot an interactive window with the park centroids and the neighborhoods
- hints: 
    - remember the shapely methods are available for each geometry object (e.g. `centroid()`) 
    - you can loop over the column in a couple different ways
    - you can define which columns contains the geometry of a geodataframe
    - you will likely have to define the CRS

## GEOSPATIAL OPERATIONS
***

### Operations on and among geodataframes...do I need to use a GIS program?

### Dissolve

In [ ]:
hoods.dissolve() # note it defaults to filling all the columns with the first value

In [ ]:
ax=hoods.dissolve().plot()
hoods.plot(facecolor=None, edgecolor='orange', ax=ax)

### Convex Hull

In [ ]:
ax = hoods.dissolve().convex_hull.plot()
hoods.plot(facecolor=None, edgecolor='orange', ax=ax)

### Bounding Box is a little more tricky

In [ ]:
hoods.bounds # that's per each row

In [ ]:
tb = hoods.total_bounds # this gives overall bounds
tb

### We can make a polygon from these coordinates with `shapely`

In [ ]:
from shapely.geometry import box

In [ ]:
bbox = box(tb[0], tb[1], tb[2], tb[3])
# pro tip - when passing a bunch of ordered arguments, '*' will unpack them #nice
bbox = box(*hoods.total_bounds)

#### to make a GeoDataFrame from scratch, the minimum you need is geometry, but a crs is important, and some data will populate more columns

In [ ]:
hoods_boundary = gp.GeoDataFrame(data={'thing':['bounding_box']},geometry=[bbox], crs=hoods.crs)
hoods_boundary

In [ ]:
ax = hoods_boundary.plot()
hoods.plot(facecolor=None, edgecolor='orange', ax=ax)

## How about some spatial joins?
***

#### we can bring in information based on locational overlap. Let's look at just a couple neighborhoods (Marquette and Tenny-Lapham) on the Isthmus

In [ ]:
isthmus = hoods.loc[hoods['NEIGHB_NAME'].str.contains('Marquette') | 
                   hoods['NEIGHB_NAME'].str.contains('Tenney')]
isthmus

In [ ]:
isthmus.explore()

In [ ]:
isthmus.sjoin(parks).explore()

In [ ]:
parks.sjoin(isthmus).explore()

#### so, it matters which direction you join from. The geometry is preserved from the dataframe "on the left"
#### equivalently, you can be more explicit in calling `sjoin`

In [ ]:
gp.sjoin(left_df=parks, right_df=isthmus).explore()

In [ ]:
isthmus_parks = gp.sjoin(left_df=parks, right_df=isthmus)

#### we are going to use this `isthmus_parks` geoDataFrame a little later, but we want to trim out some unneeded and distracting columns. We can use `.drop()` just like with a regular Pandas DataFrame

In [ ]:
isthmus_parks.columns

In [ ]:
isthmus_parks.drop(columns=[ 'index_right','OBJECTID_right', 'NA_ID', 
       'STATUS', 'CLASSIFICA', 'Web',
       'ShapeSTArea', 'ShapeSTLength'], inplace=True)

### Let's explore the various predicates with a small intersecting box

In [ ]:
bbox = box(570600, 290000, 573100, 291700)
bounds = gp.GeoDataFrame(geometry=[bbox],crs=parks.crs)
bounds.plot()

#### See [documentation](https://shapely.readthedocs.io/en/latest/manual.html#binary-predicates) for full set of options for predicates. We'll just check out a couple options: `intersects`, `contains`, `within`

## TEST YOUR SKILLS #1
***
Using the `bounds` geodataframe you just made, write a function to visualize predicate behaviors.
- your function should accept a left geodataframe, a right geodataframe, and a string for the predicate
- your function should plot:
    - the left geodataframe in (default) blue
    - the result of the spatial join operation in another color
    - the right geodataframe in another color with outline only
- then you should set the title of the plot to the string predicate value used
- the geodataframes to test with are `isthmus_parks` and `bounds`
- your function should `return` the joined geodataframe

- a couple hints:
    - in the `plot` method are a couple args called `facecolor` and `edgecolor` that will help plot the rectangle
    - there are other predicates to try out 

- _advanced options_: if that was easy, you can try a couple other things like:
    - explore joins with points and lines in addition to just polygons
    - change around the `bounds` polygon dimensions 
    - use `explore()` to make an interactive map

#### Spatial joins are particularly useful with collections of points. A common case is to add a polygon attribute to points falling within each polygon. Let's check out a bigger point dataset with all the trees on streets in Madison

In [ ]:
trees = gp.read_file(datapath / 'Street_Trees.geojson')
trees.plot(column='SPECIES')

#### let's put this into the same crs as neighborhoods and join the data together so we can have a neighborhood attribute on the trees geodataframe

In [ ]:
trees.to_crs(hoods.crs, inplace=True)

In [ ]:
hoods.columns

#### NOTE: if we pass only some columns of the GeoDataFrame, only those columns will be included in the result, which is cool. _But_ - must include the active geometry column as well!

In [ ]:
trees_with_hoods = trees[['SPECIES','DIAMETER','geometry']].sjoin(hoods[['NEIGHB_NAME','geometry']])
trees_with_hoods

#### now we can do a `groupby`, for example, to find things like the average or max diameter of trees in each neighborhood

In [ ]:
trees_with_hoods.groupby('NEIGHB_NAME')['DIAMETER'].max().plot(kind='bar')

#### We could rearrange that bar chart in various ways, but we can also flip this back to the original neighborhoods GeoDataFrame to make a more useful spatial plot. Note that we used the spatial join to join together the attribute "Neighborhood Name" with each tree point. But now, we can aggregate those results and assign them based on an attribute rather than geospatially just like regular Pandas DataFrames

In [ ]:
hood_trees = hoods.copy()
tree_summary = trees_with_hoods.groupby('NEIGHB_NAME')['DIAMETER'].max()
hood_trees.merge(tree_summary,
                left_on = 'NEIGHB_NAME', right_on='NEIGHB_NAME').explore(column="DIAMETER")

## Overlays
***
### As we've seen, spatial joins are powerful, but they really only gather data from multiple collections. What if we want to actually calculate the amount of overlap among shapes? Or create new shapes based on intersection or not intersection of shapes?

### [overlay](https://geopandas.org/en/stable/docs/user_guide/set_operations.html?highlight=overlay) does these things.

#### main options are `intersection`, `difference`, `union`, and `symmetric_difference`

In [ ]:
inters = bounds.overlay(isthmus, how='intersection')
inters.plot()

In [ ]:
inters = bounds.overlay(isthmus, how='intersection')
inters.plot()

In [ ]:
inters

In [ ]:
differ = bounds.overlay(isthmus, how='difference')
differ.plot()

In [ ]:
differ

In [ ]:
unite = bounds.overlay(isthmus, how='union')
unite.plot()

In [ ]:
unite

In [ ]:
symdiff = bounds.overlay(isthmus, how='symmetric_difference')
symdiff.plot()

In [ ]:
symdiff

### On your own...
- what if you switch left and right dataframes?
- how can you evaluate the areas of overlap for the intersection case?

## TEST YOUR SKILLS _OPTIONAL_
***
We have an Excel file that contains a crosswalk between SPECIES number as provided and species name. Can we bring that into our dataset and evaluate some conclusions about tree species by neighborhood?
- start with the `trees_with_hoods` GeoDataFrame
- load up and join the data from datapath / 'Madison_Tree_Species_Lookup.xlsx'
- hint: check the dtypes before merging - if you are going to join on a column, the column must be the same dtype in both dataframes
- Make a multipage PDF with a page for each neighborhood showing a bar chart of the top ten tree species (by name) in each neighborhood
- Make a map (use explore, or save to SHP or geojson) showing the neighborhoods with a color-coded field showing the most common tree species for each neighborhood

You will need a few pandas operations that we have only touched on a bit:  

`groupby`, `count`, `merge`, `read_excel`, `sort_values`, `iloc`

---

## 🤖 AI Sidebar: Spatial Operations with AI Assistance

*This sidebar demonstrates how an AI coding assistant approaches spatial data operations in hydrology. All AI-generated code below was captured ahead of time; no live AI calls are made.*

**Time estimate:** ~5-8 minutes

### The Task

Throughout this notebook, you worked with GeoPandas — reading spatial data, reprojecting between coordinate reference systems, performing spatial joins, and computing overlays. These are essential operations for any spatial analysis in hydrology.

Now let's see how an AI assistant handles a common hydrology task that relies on GeoPandas: **performing a spatial join of monitoring wells with aquifer boundary polygons** to determine which aquifer each well taps into. This is a routine step in groundwater studies — you have a set of well locations (points) and a map of aquifer boundaries (polygons), and you need to assign each well to its aquifer.

### AI Approach

**Prompt used:**

> *"Using GeoPandas and Shapely, write a function called `assign_wells_to_aquifers` that performs a spatial join of well locations with aquifer boundary polygons. The function should accept two GeoDataFrames: `wells` (point geometries with columns 'well_id' and 'depth_m') and `aquifers` (polygon geometries with columns 'aquifer_name' and 'aquifer_type'). Ensure both GeoDataFrames are in the same CRS before joining. Return a GeoDataFrame with the well points and their assigned aquifer attributes. Include a docstring."*

**AI-generated code:**

In [ ]:
# AI-GENERATED CODE (pre-captured)
import geopandas as gpd
from shapely.geometry import Point, Polygon

def assign_wells_to_aquifers(wells, aquifers):
    """Assign each well to an aquifer based on spatial location.

    Parameters
    ----------
    wells : geopandas.GeoDataFrame
        Point geometries with 'well_id' and 'depth_m' columns.
    aquifers : geopandas.GeoDataFrame
        Polygon geometries with 'aquifer_name' and 'aquifer_type' columns.

    Returns
    -------
    geopandas.GeoDataFrame
        Wells with aquifer attributes joined spatially.
    """
    # Reproject aquifers to match wells CRS
    if wells.crs != aquifers.crs:
        aquifers = aquifers.to_crs(wells.crs)

    # Spatial join: assign aquifer attributes to each well point
    wells_with_aquifers = gpd.sjoin(
        left_df=wells,
        right_df=aquifers,
        how='left',
        predicate='within'
    )

    # Drop the index_right column added by sjoin
    if 'index_right' in wells_with_aquifers.columns:
        wells_with_aquifers.drop(columns=['index_right'], inplace=True)

    return wells_with_aquifers

Let's test the AI-generated function with some synthetic well and aquifer data:

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon

# Create synthetic aquifer boundaries (two adjacent aquifers)
aquifer_a = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])
aquifer_b = Polygon([(10, 0), (20, 0), (20, 10), (10, 10)])

aquifers = gpd.GeoDataFrame({
    'aquifer_name': ['Sandstone Aquifer', 'Alluvial Aquifer'],
    'aquifer_type': ['confined', 'unconfined']
}, geometry=[aquifer_a, aquifer_b], crs='EPSG:3071')

# Create synthetic well locations
wells = gpd.GeoDataFrame({
    'well_id': ['MW-01', 'MW-02', 'MW-03', 'MW-04'],
    'depth_m': [45.0, 30.0, 60.0, 25.0]
}, geometry=[
    Point(3, 5),   # inside Sandstone Aquifer
    Point(15, 5),  # inside Alluvial Aquifer
    Point(8, 8),   # inside Sandstone Aquifer
    Point(25, 5),  # outside both aquifers
], crs='EPSG:3071')

# Run the spatial join
result = assign_wells_to_aquifers(wells, aquifers)
print(result[['well_id', 'depth_m', 'aquifer_name', 'aquifer_type']].to_string(index=False))
print(f"\nWells with no aquifer assigned: {result['aquifer_name'].isna().sum()}")
print("(MW-04 is outside both aquifer boundaries, so it gets NaN — that's correct!)")

### Comparison

| Aspect | Manual Approach | AI Approach |
|--------|----------------|-------------|
| **Time** | 5-10 min to set up CRS checks, choose predicate, handle edge cases | ~30 seconds to generate from a prompt |
| **Understanding** | You think through which predicate to use (`within` vs. `intersects`) and why | You get working code but may not notice the predicate choice |
| **Correctness** | You verify CRS alignment and check wells outside boundaries | AI handles the CRS check but may not warn about unmatched wells |
| **Edge cases** | You consider wells on boundaries, wells outside all polygons, duplicate matches | AI often skips handling of wells that fall on polygon edges or outside all polygons |

### What Could Go Wrong

Spatial operations are particularly sensitive to two issues that AI-generated code often gets wrong: **CRS mismatches** and **incorrect spatial join predicates**.

In [ ]:
# Problem 1: CRS mismatch — wells in lat/lon, aquifers in projected coordinates
# If the AI skips the CRS check, the join silently produces wrong results.

wells_latlon = wells.to_crs('EPSG:4326')  # convert wells to lat/lon
print(f"Wells CRS: {wells_latlon.crs}")
print(f"Aquifers CRS: {aquifers.crs}")
print()

# Spatial join WITHOUT reprojecting first
try:
    bad_result = gpd.sjoin(wells_latlon, aquifers, how='left', predicate='within')
    n_matched = bad_result['aquifer_name'].notna().sum()
    print(f"Wells matched (no CRS fix): {n_matched} out of {len(wells_latlon)}")
    print("-> No wells matched because the coordinates are in completely different systems!")
    print("   Wells are in degrees (lat/lon), aquifers are in meters (EPSG:3071).")
except Exception as e:
    print(f"Error: {e}")
    print("-> GeoPandas raised an error because the CRS values don't match.")

print()
print("Key lesson: ALWAYS check that both GeoDataFrames share the same CRS")
print("before performing spatial operations. Use .to_crs() to reproject.")

In [ ]:
# Problem 2: Wrong spatial join predicate
# 'intersects' vs 'within' can produce very different results.

# Place a well exactly on the boundary between the two aquifers
boundary_well = gpd.GeoDataFrame({
    'well_id': ['MW-05'],
    'depth_m': [35.0]
}, geometry=[Point(10, 5)], crs='EPSG:3071')  # right on the shared edge

# Using 'within' — strict containment
result_within = gpd.sjoin(boundary_well, aquifers, how='left', predicate='within')
print("Using predicate='within':")
print(f"  Matches: {result_within['aquifer_name'].notna().sum()}")
print(f"  Aquifer(s): {result_within['aquifer_name'].tolist()}")
print()

# Using 'intersects' — includes boundary touches
result_intersects = gpd.sjoin(boundary_well, aquifers, how='left', predicate='intersects')
print("Using predicate='intersects':")
print(f"  Matches: {result_intersects['aquifer_name'].notna().sum()}")
print(f"  Aquifer(s): {result_intersects['aquifer_name'].tolist()}")
print()
print("-> 'intersects' may match BOTH aquifers for a well on the boundary,")
print("   creating duplicate rows. 'within' may match neither.")
print("   Choose the predicate carefully based on your analysis needs!")

**Key takeaway:** In hydrology, CRS errors are one of the most common and dangerous spatial bugs. A spatial join between datasets in different CRS values will either fail or silently return no matches. AI-generated code sometimes includes a CRS check (as in our example above) but often does not. Additionally, the choice of spatial join predicate (`within`, `intersects`, `contains`) matters — `intersects` can produce duplicate matches for features on polygon boundaries, while `within` may miss them entirely. Always verify the number of matched and unmatched features after a spatial join.

### When to Use AI Here

- ✅ **Well-suited for AI:** Generating boilerplate for reading shapefiles, GeoJSON, and other spatial formats with `gp.read_file()`
- ✅ **Well-suited for AI:** Writing spatial join and overlay syntax — the API has many parameters and AI can save lookup time
- ✅ **Well-suited for AI:** Creating quick visualizations with `.plot()` and `.explore()`, including styling options
- ⚠️ **Requires manual understanding:** Choosing the correct CRS for your study area and verifying that all layers are aligned
- ⚠️ **Requires manual understanding:** Selecting the appropriate spatial join predicate (`within`, `intersects`, `contains`) for your analysis question
- ⚠️ **Requires manual understanding:** Interpreting results when wells fall outside all polygons or on polygon boundaries

---

### 🤖 AI Exercise Variant: Spatial Join of Wells with Aquifer Boundaries

Now it's your turn to use an AI assistant (or imagine using one) to perform a spatial analysis task.

**Your task:** Use an AI coding assistant to write a function called `summarize_wells_by_aquifer` that:
1. Accepts two GeoDataFrames: `wells` (point geometries with columns `'well_id'`, `'depth_m'`, and `'yield_lpm'` for yield in liters per minute) and `aquifers` (polygon geometries with columns `'aquifer_name'` and `'aquifer_type'`)
2. Ensures both GeoDataFrames are in the same CRS (reproject if needed)
3. Performs a spatial join to assign each well to an aquifer
4. Returns a Pandas DataFrame (not GeoDataFrame) summarizing each aquifer with: number of wells, mean depth (m), and mean yield (lpm)

**Ground truth values for verification:**

Using these inputs:
- Aquifer A (polygon covering x=0-10, y=0-10): `'Sandstone'`, `'confined'`
- Aquifer B (polygon covering x=10-20, y=0-10): `'Alluvial'`, `'unconfined'`
- Well MW-01 at (3, 5): depth=45.0 m, yield=120.0 lpm → Sandstone
- Well MW-02 at (15, 5): depth=30.0 m, yield=200.0 lpm → Alluvial
- Well MW-03 at (8, 8): depth=60.0 m, yield=80.0 lpm → Sandstone

Expected summary:
- Sandstone: 2 wells, mean depth = 52.5 m, mean yield = 100.0 lpm
- Alluvial: 1 well, mean depth = 30.0 m, mean yield = 200.0 lpm

**Step 1:** Write your prompt in the cell below.

In [ ]:
# Write your prompt here as a comment or string:
my_prompt = """
YOUR PROMPT HERE
"""

**Step 2:** Paste or write the AI-generated function in the cell below, then run it.

In [ ]:
# Paste your AI-generated summarize_wells_by_aquifer function here:



**Step 3:** Verify your function against the ground truth values.

In [ ]:
# Verification tests — uncomment and fill in the function calls
#
# import geopandas as gpd
# from shapely.geometry import Point, Polygon
#
# # Set up test data
# aquifer_a = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])
# aquifer_b = Polygon([(10, 0), (20, 0), (20, 10), (10, 10)])
# test_aquifers = gpd.GeoDataFrame({
#     'aquifer_name': ['Sandstone', 'Alluvial'],
#     'aquifer_type': ['confined', 'unconfined']
# }, geometry=[aquifer_a, aquifer_b], crs='EPSG:3071')
#
# test_wells = gpd.GeoDataFrame({
#     'well_id': ['MW-01', 'MW-02', 'MW-03'],
#     'depth_m': [45.0, 30.0, 60.0],
#     'yield_lpm': [120.0, 200.0, 80.0]
# }, geometry=[Point(3, 5), Point(15, 5), Point(8, 8)], crs='EPSG:3071')
#
# summary = summarize_wells_by_aquifer(test_wells, test_aquifers)
# print(summary)
# print()
#
# # Check expected values
# sandstone = summary[summary['aquifer_name'] == 'Sandstone']
# print(f"Sandstone well count: {sandstone['well_count'].values[0]}  (expected: 2)")
# print(f"Sandstone mean depth: {sandstone['mean_depth_m'].values[0]:.1f}  (expected: 52.5)")
# print(f"Sandstone mean yield: {sandstone['mean_yield_lpm'].values[0]:.1f}  (expected: 100.0)")

### ✅ Verification Checklist

- [ ] `assign_wells_to_aquifers` function runs without errors and correctly assigns wells to aquifers
- [ ] Output matches Ground Truth: Sandstone has 2 wells (mean depth 52.5 m, mean yield 100.0 lpm); Alluvial has 1 well (mean depth 30.0 m, mean yield 200.0 lpm)
- [ ] Domain review: CRS is consistent between wells and aquifers before the spatial join
- [ ] Domain review: wells outside all aquifer boundaries are handled (NaN values, not silently dropped)
- [ ] Code quality: correct spatial join predicate chosen, no unnecessary complexity, includes a docstring

### Prompt Log

In [ ]:
# Prompt Log
# ----------
# Prompt version 1: (paste your first prompt here)
#
# Result: (did the AI-generated code work? what was wrong?)
#
# Prompt version 2 (if needed): (paste your revised prompt here)
#
# Result: (did the revision fix the issue?)
#
# Number of iterations to reach correct result: 
#
# What did you learn about prompting for spatial operations?